# Record-cleaning: Phase 1 localization series (07c–07h)

Run-once, **idempotent, append-only**. It does not overwrite any existing finding, decision,
or map entry — it appends clearly-marked corrections/annotations and one consolidated
journal entry, each guarded by a marker so re-running is a no-op. This preserves the audit
trail (the original auto-logged verdicts stand as the run record; these are the corrected
interpretations).

What it writes:
1. **F007 correction** (findings_log) — 07c "SHAP-ADDS-VALUE" was artifact-sensitive; GT3
   (permutation) was modality-contaminated; corrected = REDUNDANT / KS-competitive.
2. **F011 correction** (findings_log) — 07g real-low arm invalid (NaN recall from constant
   features); the bankable 07g result is the *synthetic* mechanism confirmation.
3. **F012 caveat** (findings_log) — 07h COMPOUND-STRUCTURAL-BLOCK holds, but the corrected
   arm stayed confounded and ran anti-mechanism, so no validated real-data diagnostic.
4. **Regime-map resolution** (drift_regime_map.md) — governing-variable row resolved with
   the softened, honest framing.
5. **Journal** — one consolidated 07c–07h verdict log + state-of-project summary.

It also prints the exact one-line replacement for the regime-map §8 governing-variable row,
in case you prefer to edit that row in place rather than rely on the appended resolution.

In [1]:
import datetime
from pathlib import Path
import sys
sys.path.insert(0, '/content/drive/MyDrive/phd_thesis')
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    pass
from src.utils.documentation import add_journal_entry

ROOT = Path('/content/drive/MyDrive/phd_thesis')
NOTES = ROOT / 'reports' / 'advisor_notes'
FINDINGS = NOTES / 'findings_log.md'
REGIME_MAP = NOTES / 'drift_regime_map.md'
JOURNAL = NOTES / 'thesis_journal.md'
TODAY = datetime.date.today().isoformat()

def append_if_absent(path, marker, block):
    "Append block to path only if marker is not already present. Idempotent."
    path.parent.mkdir(parents=True, exist_ok=True)
    existing = path.read_text() if path.exists() else ''
    if marker in existing:
        print(f'[SKIP] {path.name}: "{marker[:42]}..." already present'); return False
    with open(path, 'a') as f:
        if existing and not existing.endswith('\n'):
            f.write('\n')
        f.write(block)
    print(f'[APPENDED] {path.name}: {marker[:42]}...'); return True

# 1) F007 correction --------------------------------------------------------- #
F007_MARK = '## F007 \u2014 CORRECTION (post-07d)'
F007_BLOCK = f'''
{F007_MARK}  ({TODAY})
**Applies to:** F007 (07c, KS vs SHAP localization). Annotation, not overwrite.

The cell's auto-verdict "SHAP-ADDS-VALUE" was artifact-sensitive and is corrected to
**REDUNDANT / KS-competitive**. It rested on GT3 (oracle *permutation* importance), which
07d showed to be the unstable estimator (permutation self-stability 0.584 vs SHAP 0.921;
SHAP and gain agree at 0.543 while permutation agrees with neither, ~0.15). GT3 was
therefore modality-matched to the permutation baseline, not the neutral arbiter it was
treated as. On GT2 (oracle SHAP) a label-free **KS test (0.833) beat aux-SHAP (0.766)**,
and GT2 vs GT3 overlapped at Jaccard 0.11-0.20 (~chance). Corrected reading: SHAP
localization does **not** robustly beat a free KS test on CICIDS.
'''

# 2) F011 correction --------------------------------------------------------- #
F011_MARK = '## F011 \u2014 CORRECTION (post-07h)'
F011_BLOCK = f'''
{F011_MARK}  ({TODAY})
**Applies to:** F011 (07g, decorrelated-limit + synthetic mechanism). Annotation, not overwrite.

The auto-verdict "MECHANISM-HOLDS-SYNTH-ONLY" over-read a broken construction. The real
low-redundancy arm returned **aux_recall = NaN**: its features (fwd_urg_flags,
bwd_urg_flags, urg_flag_count, subflow_bwd_packets) are **constant in the Monday sample**
(distinct = 1, modal fraction 1.000), so the median-split label collapsed to a single class
(balance 0.0000) and the model never trained. Its 0.067 precision is noise, not evidence of
a "second factor."
**Bankable from 07g:** the *synthetic* identifiability mechanism is confirmed -- with a
proxy-free target aux-SHAP recovers perfectly (rho=0 precision 1.000), degrading
monotonically as proxies strengthen (rho=0.99 -> 0.533, trend +0.467), KS at chance
throughout. Real reachability is resolved in F012 (07h), not here.
'''

# 3) F012 caveat ------------------------------------------------------------- #
F012_MARK = '## F012 \u2014 CAVEAT (corrected-arm confound)'
F012_BLOCK = f'''
{F012_MARK}  ({TODAY})
**Applies to:** F012 (07h, degeneracy diagnostic + corrected real arm). Annotation.

The verdict **COMPOUND-STRUCTURAL-BLOCK holds** on its logic: of 82 features, 63 are
non-degenerate, and the lowest-redundancy non-degenerate set of size 5 has external leakage
0.248 (> 0.20), so no decorrelated-and-informative set exists; the decorrelated features are
constant/near-constant. Two caveats for honest reporting:
(1) The corrected arm **remained confounded** -- the lowest-redundancy non-degenerate
features are still the most near-constant within the band (modal 0.900, 0.875), and the
**correlated set localized better than the decorrelated set** (aux-SHAP 0.600 vs 0.267),
the opposite of the identifiability prediction. So external leakage does **not** predict
recovery on real CICIDS features, and the real arm does not validate "decorrelate ->
recover."
(2) Consequently the identifiability account **explains** the synthetic regime and the
real-data failure, but does **not** yield a validated scalar deployment diagnostic on
CICIDS. Claim the redundancy/degeneracy diagnostic as *mechanism-motivated*, not
*validated-predictive*. The clean leakage->recovery test belongs in breadth (UNSW, less
collinear), which can construct the un-confounded regime CICIDS cannot.
'''

# 4) Regime-map resolution --------------------------------------------------- #
MAP_MARK = '## Resolution update \u2014 Phase 1 localization closed (post-07h)'
MAP_BLOCK = f'''
{MAP_MARK}  ({TODAY})

**Governing-variable row (S8 ledger) -- RESOLVED, with qualification.**

Drift *type* is **not** the governing variable (07e/F009: aux-SHAP at chance in both drift
regimes on real correlated geometry, construction valid). The candidate was feature
correlation / target identifiability. Resolution across 07f-07h:

- **Controlled (synthetic) setting:** explanation-localization recovery **is** governed by
  target identifiability (proxy availability) -- no proxies -> recovery (precision 1.000),
  proxies -> monotone degradation (07g/F011). Mechanism confirmed.
- **Real CICIDS:** this variable **cannot be exercised**. The only decorrelated features are
  degenerate/constant (unlearnable) and the informative features are collinear (median
  redundancy 0.95), so **no identifiable localization target exists** (07h/F012); and among
  non-degenerate features external leakage does **not** predict recovery (still confounded
  with degeneracy; the correlated set localized better).

**Net:** identifiability explains both the synthetic success and the real-data failure, but
does **not** yield a validated, dataset-agnostic predictive diagnostic. The redundancy +
distributional-health diagnostic is *mechanism-motivated*, pending a clean leakage->recovery
test on a less-collinear dataset (UNSW) in the breadth phase. The positive-shaped search on
CICIDS is closed.
'''

for path, mark, block in [
    (FINDINGS, F007_MARK, F007_BLOCK),
    (FINDINGS, F011_MARK, F011_BLOCK),
    (FINDINGS, F012_MARK, F012_BLOCK),
    (REGIME_MAP, MAP_MARK, MAP_BLOCK),
]:
    append_if_absent(path, mark, block)

# 5) Consolidated journal entry (idempotent on title) ------------------------ #
JTITLE = 'Phase 1 localization series (07c-07h): verdicts and state of project'
JBODY = (
    'Phase 1 localization series closed. Verdicts (corrected where the auto-verdict '
    'over-read a construction):\\n'
    '- 07c/F007: KS vs SHAP. Auto "SHAP-ADDS-VALUE" CORRECTED to REDUNDANT/KS-competitive '
    '-- GT3 (permutation) was modality-contaminated (07d); on GT2 KS 0.833 beat aux-SHAP 0.766.\\n'
    '- 07d/F008: PERMUTATION-DRIVEN. The "disagreement" was permutation instability '
    '(perm self-stability 0.584 vs SHAP 0.921; SHAP-gain agree 0.543). Disagreement '
    'mechanism demoted to a methodological caveat.\\n'
    '- 07e/F009: H1 REFUTED. Drift TYPE is not the governing variable -- aux-SHAP at chance '
    'in BOTH regimes on real correlated geometry (construction valid, concept ks_max 0.017).\\n'
    '- 07f/F010: H2-REFUTED-FLAT but construction-limited -- its "low correlation" level was '
    'leakage 0.37, never decorrelated; reframed as "decorrelated regime unreachable on CICIDS".\\n'
    '- 07g/F011: Synthetic identifiability mechanism CONFIRMED (rho=0 SHAP 1.000 -> rho=0.99 '
    '0.533, trend +0.467). Real-low arm INVALID (NaN recall, constant features); corrected in 07h.\\n'
    '- 07h/F012: COMPOUND-STRUCTURAL-BLOCK. CICIDS cannot furnish an identifiable localization '
    'target -- decorrelated features are constant (fwd_urg_flags etc., distinct=1, balance '
    '0.0000), informative features collinear (median redundancy 0.95), no size-5 '
    'decorrelated-non-degenerate set (leakage floor 0.248). Caveat: corrected arm stayed '
    'confounded; correlated set localized BETTER (0.600 vs 0.267), so external leakage does '
    'NOT predict recovery on real CICIDS -- identifiability explains synthetic success + real '
    'failure but is NOT a validated predictive diagnostic.\\n\\n'
    'STATE OF PROJECT: the original dual-model SHAP localization hypothesis is fully '
    'falsified. Bankable contributions -- (C1) the pre-registered, modality-aware evaluation '
    'protocol incl. the modality-circularity (07c) and permutation-instability (07d) lessons; '
    '(C2) the distributional-regime redundancy benchmark (KS >= SHAP, dual adds nothing); '
    '(C3) the synthetic identifiability mechanism + the CICIDS structural-impossibility '
    'result; (C4) the regime map. Positive-shaped search is CLOSED. Next: breadth (UNSW + '
    'XGBoost) with per-dataset redundancy + degeneracy measurement and the clean '
    'leakage->recovery test (deferred from CICIDS, which structurally cannot run it). Still '
    'open: verify the synthetic pilot (F001) held P(X) stable during the relevance swap.'
)
if JOURNAL.exists() and JTITLE in JOURNAL.read_text():
    print(f'[SKIP] journal: entry "{JTITLE[:42]}..." already present')
else:
    add_journal_entry(title=JTITLE, body=JBODY)
    print('[JOURNAL ADDED]', JTITLE)

print('\n--- OPTIONAL in-place edit ---')
print('If you prefer to edit the regime-map S8 governing-variable row directly, replace it with:')
print('  Governing variable | RESOLVED (qualified): target identifiability (proxy '
      'availability) governs recovery in controlled settings (07g); on real CICIDS an '
      'identifiable target is unconstructable (07h) and external leakage does not predict '
      'recovery, so identifiability explains synthetic success + real failure but is NOT a '
      'validated predictive diagnostic. Clean test deferred to UNSW (breadth).')
print('\nRecord-cleaning complete.')

Mounted at /content/drive
[APPENDED] findings_log.md: ## F007 — CORRECTION (post-07d)...
[APPENDED] findings_log.md: ## F011 — CORRECTION (post-07h)...
[APPENDED] findings_log.md: ## F012 — CAVEAT (corrected-arm confound)...
[APPENDED] drift_regime_map.md: ## Resolution update — Phase 1 localizatio...
[JOURNAL ENTRY] Phase 1 localization series (07c-07h): verdicts and state of project
[JOURNAL ADDED] Phase 1 localization series (07c-07h): verdicts and state of project

--- OPTIONAL in-place edit ---
If you prefer to edit the regime-map S8 governing-variable row directly, replace it with:
  Governing variable | RESOLVED (qualified): target identifiability (proxy availability) governs recovery in controlled settings (07g); on real CICIDS an identifiable target is unconstructable (07h) and external leakage does not predict recovery, so identifiability explains synthetic success + real failure but is NOT a validated predictive diagnostic. Clean test deferred to UNSW (breadth).

Record-clea

## Done

Re-running this notebook is safe — every write is guarded by a marker, so a second run prints
`[SKIP]` for each. The still-open experimental item (verify the synthetic pilot F001 held
P(X) stable during the relevance swap) is noted in the journal entry but is a check to run,
not a record edit, so it is not performed here.